In [9]:
# Step 1: Import Dependencies
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import GaussianNB

In [10]:
# Step 2: Load the dataset
df = pd.read_csv("titanic.csv")
print(df.head())

   PassengerId  Survived  Pclass       Name      Sex   Age  SibSp  Parch  \
0            1         0       3     Braund     male  22.0      1      0   
1            2         1       1    Cumings   female  38.0      1      0   
2            3         1       3  Heikkinen   female  26.0      0      0   
3            4         1       1   Futrelle   female  35.0      1      0   
4            5         0       3      Allen     male  35.0      0      0   

             Ticket     Fare Cabin Embarked  
0         A/5 21171   7.2500   NaN        S  
1          PC 17599  71.2833   C85        C  
2  STON/O2. 3101282   7.9250   NaN        S  
3            113803  53.1000  C123        S  
4            373450   8.0500   NaN        S  


In [11]:
# Step 3: Split the features and target
inputs = df.drop(['PassengerId','Name','SibSp','Parch','Ticket','Cabin','Embarked','Survived'], axis='columns')
target = df['Survived']

In [12]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,Braund,male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,Cumings,female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,Heikkinen,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,Futrelle,female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,Allen,male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,Moran,male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,McCarthy,male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,Palsson,male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,Johnson,female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,Nasser,female,14.0,1,0,237736,30.0708,NaN,C


In [13]:
# Step 4: Handle categorical data (Sex → one-hot encoding)
dummies = pd.get_dummies(inputs.Sex)
inputs = pd.concat([inputs, dummies], axis='columns')

# drop original Sex column
inputs.drop(['Sex'], axis='columns', inplace=True, errors='ignore')

# drop any "female" dummy column (whatever name exists)
for col in inputs.columns:
    if 'female' in col.lower():
        inputs.drop([col], axis='columns', inplace=True)
        break  # drop only one dummy


In [14]:
dummies

,female,male
0,False,True
1,True,False
2,True,False
3,True,False
4,False,True
5,False,True
6,False,True
7,False,True
8,True,False
9,True,False


In [15]:
inputs

,Pclass,Age,Fare,male
0,3,22.0,7.2500,True
1,1,38.0,71.2833,False
2,3,26.0,7.9250,False
3,1,35.0,53.1000,False
4,3,35.0,8.0500,True
5,3,NaN,8.4583,True
6,1,54.0,51.8625,True
7,3,2.0,21.0750,True
8,3,27.0,11.1333,False
9,2,14.0,30.0708,False


In [16]:
# Step 5: Handle missing values (fill Age with mean)
inputs.Age = inputs.Age.fillna(inputs.Age.mean())

In [17]:
inputs

,Pclass,Age,Fare,male
0,3,22.0,7.2500,True
1,1,38.0,71.2833,False
2,3,26.0,7.9250,False
3,1,35.0,53.1000,False
4,3,35.0,8.0500,True
5,3,28.0,8.4583,True
6,1,54.0,51.8625,True
7,3,2.0,21.0750,True
8,3,27.0,11.1333,False
9,2,14.0,30.0708,False


In [18]:
# Step 6: Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(inputs, target, test_size=0.3)

In [19]:
# Step 7: Train Gaussian Naïve Bayes model
model = GaussianNB()
model.fit(X_train, y_train)


GaussianNB()

In [20]:
# Step 8: Find accuracy on test data
print("Test Accuracy:", model.score(X_test, y_test))

Test Accuracy: 1.0


In [21]:
# Step 9: Show some predictions vs actual
print("Actual values (first 20):")
print(y_test[:20].values)

print("Predicted values (first 20):")
print(model.predict(X_test[:20]))

Actual values (first 20):
[1 1 0 0 0 1]
Predicted values (first 20):
[1 1 0 0 0 1]


In [22]:
# Step 10: Show prediction probabilities
print("Prediction Probabilities (first 10):")
print(model.predict_proba(X_test[:10]))

Prediction Probabilities (first 10):
[[3.28027171e-04 9.99671973e-01]
 [3.91529649e-05 9.99960847e-01]
 [1.00000000e+00 0.00000000e+00]
 [1.00000000e+00 0.00000000e+00]
 [1.00000000e+00 0.00000000e+00]
 [1.01273848e-03 9.98987262e-01]]


In [23]:
# Step 11: Perform Cross Validation
scores = cross_val_score(model, inputs, target, cv=5)
print("Cross Validation Scores:", scores)
print("Average CV Score:", scores.mean())

Cross Validation Scores: [1.   0.5  1.   0.75 0.75]
Average CV Score: 0.8
